In [26]:
import sys
import os
import logging
from collections import Counter
# pyrefly: ignore [missing-import]
from datasets import load_dataset, DatasetDict, concatenate_datasets
from typing import List
sys.path.append(os.path.abspath('..'))
os.chdir('..')
from transformers import AutoTokenizer

from src.utils.config import load_config
from src.data.preprocessor import MultilingualPreprocessor

In [15]:
data_config = load_config("configs/data_config.yaml")
model_config = load_config("configs/model_config.yaml")
cache_dir = "./data/raw"
max_samples = data_config['max_samples']
val_samples = data_config['val_samples']
test_samples = data_config['test_samples']

In [4]:
print("\n2. Bắt đầu tải OPUS-100 với các tham số:")
print(f"- Các cặp ngôn ngữ: {data_config['lang_pairs']}")
print(f"- Max samples (train): {data_config['max_samples']}")
print(f"- Val samples: {data_config['val_samples']}")
print(f"- Test samples: {data_config['test_samples']}")


2. Bắt đầu tải OPUS-100 với các tham số:
- Các cặp ngôn ngữ: ['en-vi', 'en-fr', 'de-en']
- Max samples (train): 500
- Val samples: 10
- Test samples: 20


In [12]:
lang_pair = data_config['lang_pairs']
all_train, all_val, all_test = [], [], []
for pair in lang_pair:
    src, tgt = pair.split('-')

    langs = sorted([src, tgt])
    config_name = f"{langs[0]}-{langs[1]}"
    ds = load_dataset("Helsinki-NLP/opus-100", config_name, cache_dir=cache_dir)

    train = ds["train"].select(range(min(len(ds["train"]), max_samples)))
    val = ds["validation"] if "validation" in ds else ds["test"].select(range(min(len(ds["test"]), val_samples)))
    test = ds["test"].select(range(min(len(ds["test"]), test_samples)))

    train = train.map(lambda x: {"pair": pair, "src": x["translation"][src], "tgt": x["translation"][tgt]}, remove_columns=["translation"])
    val = val.map(lambda x: {"pair": pair, "src": x["translation"][src], "tgt": x["translation"][tgt]}, remove_columns=["translation"])
    test = test.map(lambda x: {"pair": pair, "src": x["translation"][src], "tgt": x["translation"][tgt]}, remove_columns=["translation"])

    all_train.append(train)
    all_val.append(val)
    all_test.append(test)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [31]:
dataset = DatasetDict({
    "train": concatenate_datasets(all_train).shuffle(seed=42),
    "validation": concatenate_datasets(all_val),
    "test": concatenate_datasets(all_test)
})

# In phân phối dữ liệu
pair_counts = Counter(dataset["train"]["pair"])

In [29]:
model_name = model_config["model_name"]
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

d:\conda_envs\env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub\models--facebook--mbart-large-50-many-to-many-mmt. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

In [30]:
preprocessor = MultilingualPreprocessor(tokenizer, data_config['max_length'])

In [32]:
tokenized_dataset = dataset.map(
    preprocessor.preprocess_function, 
    batched=True,       
    batch_size=100
)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]